# CSPF — leave-one-family-out (یک سلول)
می‌پرسد: آیا آموزش consistency روی ۱۱ خانواده، پایداری روی خانواده‌ی دوازدهمِ دیده‌نشده را بهتر می‌کند؟ زمان: حدود ۵۰ دقیقه روی T4؛ نتایج روی Drive ذخیره می‌شوند و بعد از قطع شدن ادامه می‌دهد.

In [ ]:
# CSPF — leave-one-family-out consistency training (ONE CELL). Everything is saved to Drive; re-run after a disconnect to resume.
# Needs: ALL_MODELS_combined_FIXED.csv in BASE, and the previous consistency runs (results_FIXED/consistency/runs/erm_s*.npz, cr_all_s*.npz).
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
import subprocess, sys; subprocess.run([sys.executable,"-m","pip","-q","install","sentencepiece"],check=False)

BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
OUT_DIR   = f"{BASE}/results_FIXED/consistency"   # same folder: reuses the erm / cr_all runs already there
BACKBONE  = "distilroberta-base"
METHODS   = []
SEEDS     = [0, 1, 2]
LAMBDA    = 1.0                  # weight of the consistency / augmentation term
EPOCHS, BS, LR, MAXLEN, WARMUP = 2, 32, 2e-5, 96, 0.06
SPLIT_SEED, EVAL_CAP = 0, 6000
PROP = ["P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"]
import os; os.makedirs(OUT_DIR+"/runs", exist_ok=True)


import pandas as pd, numpy as np, os, glob, time, warnings; warnings.filterwarnings("ignore")
cols = ["source_dataset","review_idx","original_review","perturbed_review","sentiment","perturbation_family"]
df = pd.read_csv(FIXED_CSV, usecols=cols, low_memory=False)
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str)
df["y"]   = (df.sentiment=="positive").astype(np.int64)
FAMS = sorted(df.perturbation_family.unique())
U = df.drop_duplicates("key")[["key","source_dataset","original_review","y"]].reset_index(drop=True)
assert U.y.isin([0,1]).all() and (df.groupby("key").y.nunique()==1).all()
rng = np.random.default_rng(SPLIT_SEED); perm = rng.permutation(len(U)); n=len(U)
U["split"] = "train"; U.loc[perm[:int(.20*n)],"split"] = "test"; U.loc[perm[int(.20*n):int(.25*n)],"split"] = "dev"
df["split"] = df.key.map(U.set_index("key").split)
print(len(df),"pairs |",n,"reviews |", U.split.value_counts().to_dict())
print(df.groupby("split").perturbation_family.value_counts().unstack(0).fillna(0).astype(int))


# ── fixed evaluation set (identical for every run): all test originals + capped, re-weighted test pairs ──
test_pairs = df[df.split=="test"]
sh = test_pairs.sample(frac=1, random_state=SPLIT_SEED)
EP = sh[sh.groupby("perturbation_family").cumcount() < EVAL_CAP].sort_index().reset_index(drop=True)
fam_tot = test_pairs.groupby("perturbation_family").size(); fam_n = EP.groupby("perturbation_family").size()
EP["w"] = EP.perturbation_family.map(fam_tot/fam_n).astype(float)                     # corpus-proportional weights
EO = U[U.split=="test"].reset_index(drop=True)
key2o = pd.Series(np.arange(len(EO)), index=EO.key); EP["o_idx"] = EP.key.map(key2o).values
print(len(EO),"test originals;",len(EP),"evaluation pairs (weights sum =",round(EP.w.sum()),")")

# ── training pools: for every train review, the pair indices usable by each method ──
TR = U[U.split=="train"].reset_index(drop=True)
trp = df[df.split=="train"].reset_index(drop=True)
def pools(fams):
    sub = trp[trp.perturbation_family.isin(fams)]
    return {k: v for k, v in sub.groupby("key").indices.items()}, sub.perturbed_review.values
POOLS = {"prop": pools(PROP), "all": pools(FAMS)}
print("train reviews:", len(TR), "| with a propositional pair:", len(POOLS["prop"][0]), "| with any pair:", len(POOLS["all"][0]))
METHOD_CFG = {   # name: (pool, loss on perturbed)
  "erm": (None, None), "aug_prop": ("prop","ce"), "cr_prop": ("prop","kl"), "cr_all": ("all","kl"), "aug_all": ("all","ce")}


import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
dev = "cuda" if torch.cuda.is_available() else "cpu"; print(dev)
tok = AutoTokenizer.from_pretrained(BACKBONE)

def enc(texts): return tok(list(texts), padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)

@torch.no_grad()
def predict(model, texts, bs=256):
    model.eval(); out = np.zeros(len(texts), dtype=np.int64); order = np.argsort([len(t) for t in texts])
    for i in range(0, len(order), bs):
        ix = order[i:i+bs]
        with torch.autocast(dev, dtype=torch.float16, enabled=(dev=="cuda")):
            out[ix] = model(**enc([texts[j] for j in ix])).logits.argmax(-1).cpu().numpy()
    return out

def sym_kl(lp, lq):   # log-prob inputs, per-example symmetric KL
    p, q = lp.exp(), lq.exp()
    return 0.5*((p*(lp-lq)).sum(-1) + (q*(lq-lp)).sum(-1))

def train(method, seed):
    torch.manual_seed(seed); rs = np.random.default_rng(seed)
    pool_name, ploss = METHOD_CFG[method]
    model = AutoModelForSequenceClassification.from_pretrained(BACKBONE, num_labels=2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(TR)/BS)); sch = get_linear_schedule_with_warmup(opt, int(WARMUP*steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev=="cuda"))
    pool, ptexts = POOLS[pool_name] if pool_name else ({}, None)
    texts, ys, keys = TR.original_review.values, TR.y.values, TR.key.values
    for ep in range(EPOCHS):
        model.train(); order = rs.permutation(len(TR)); t0 = time.time(); tot = 0.0
        for s in range(0, len(order), BS):
            ix = order[s:s+BS]; x = [texts[i] for i in ix]; y = torch.tensor(ys[ix], device=dev)
            if pool_name:   # one random usable perturbation per review (mask = review has one)
                pj = [(pool[keys[i]][rs.integers(len(pool[keys[i]]))] if keys[i] in pool else -1) for i in ix]
                m = np.array([j >= 0 for j in pj]); xp = [ptexts[j] if j >= 0 else texts[i] for i, j in zip(ix, pj)]
            with torch.autocast(dev, dtype=torch.float16, enabled=(dev=="cuda")):
                if pool_name:
                    lg = model(**enc(x + xp)).logits.float(); lo, lp = lg[:len(x)], lg[len(x):]
                    mt = torch.tensor(m, device=dev, dtype=torch.float32); ce = F.cross_entropy(lo, y)
                    if ploss == "ce":  extra = (F.cross_entropy(lp, y, reduction="none")*mt).sum()/mt.sum().clamp(min=1)
                    else:              extra = (sym_kl(F.log_softmax(lo,-1), F.log_softmax(lp,-1))*mt).sum()/mt.sum().clamp(min=1)
                    loss = ce + LAMBDA*extra
                else:
                    loss = F.cross_entropy(model(**enc(x)).logits.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
        print(f"  {method} seed{seed} epoch {ep+1}: loss {tot/np.ceil(len(order)/BS):.4f} ({time.time()-t0:.0f}s)")
    return model


def evaluate(pred_o, pred_p):
    # pred_o: predictions on EO (test originals); pred_p: predictions on EP.perturbed_review
    corr_o = (pred_o == EO.y.values)
    po_pair = pred_o[EP.o_idx.values]                       # original prediction aligned to each pair
    flip = (po_pair != pred_p).astype(np.int8); oc = corr_o[EP.o_idx.values]
    w = EP.w.values; fam = EP.perturbation_family.values
    def wm(x, mask=None):
        mask = np.ones(len(x), bool) if mask is None else mask
        return float((x[mask]*w[mask]).sum()/w[mask].sum())
    propm = np.isin(fam, PROP)
    r = dict(acc_orig=float(corr_o.mean()), acc_pert=wm((pred_p == EP.y.values).astype(float)), FR=wm(flip), FRc=wm(flip, oc), FRw=wm(flip, ~oc),
             FR_prop=wm(flip, propm), FR_mod=wm(flip, ~propm))
    for f in FAMS: r["FR_"+f] = wm(flip, fam==f)
    r["DR"] = wm(((flip==1)&oc).astype(float)); r["IRR"] = wm(((flip==1)&~oc).astype(float))
    return r, flip

def run_one(method, seed):
    f = f"{OUT_DIR}/runs/{method}_s{seed}.npz"
    if os.path.exists(f): return
    t0 = time.time(); model = train(method, seed)
    pred_o = predict(model, EO.original_review.tolist()); pred_p = predict(model, EP.perturbed_review.tolist())
    dv = U[U.split=="dev"]; acc_dev = float((predict(model, dv.original_review.tolist()) == dv.y.values).mean())
    r, flip = evaluate(pred_o, pred_p); r.update(method=method, seed=seed, acc_dev=acc_dev, minutes=(time.time()-t0)/60)
    np.savez_compressed(f, flip=flip, pred_o=pred_o.astype(np.int8), **{k: np.array(v) for k, v in r.items()})
    print(f"[{method} s{seed}] acc_o {r['acc_orig']:.4f} acc_p {r['acc_pert']:.4f} FR {r['FR']:.4f} FR_prop {r['FR_prop']:.4f} FR_mod {r['FR_mod']:.4f}  ({r['minutes']:.1f} min)")
    del model; torch.cuda.empty_cache()

# ── LEAVE-ONE-FAMILY-OUT: train consistency on 11 families, test on the 12th (never seen in training) ──
LOFO_SEEDS = [0]          # add 1 for a second seed (+~50 min)
for f in FAMS:
    nm = "lofo_" + f
    POOLS[nm] = pools([x for x in FAMS if x != f]); METHOD_CFG[nm] = (nm, "kl")
for seed in LOFO_SEEDS:
    for f in FAMS:
        run_one("lofo_" + f, seed)

# ── analysis: held-out family FR vs erm (no consistency) vs cr_all (family seen in training) ──
import zipfile
w = EP.w.values; fam = EP.perturbation_family.values
keys_, inv = np.unique(EP.key.values, return_inverse=True); nk = len(keys_); rng = np.random.default_rng(11)
def load(m, ss):
    fl = [np.load(f"{OUT_DIR}/runs/{m}_s{s}.npz")["flip"] for s in ss if os.path.exists(f"{OUT_DIR}/runs/{m}_s{s}.npz")]
    return np.mean(fl, axis=0) if fl else None
erm, cra = load("erm", [0,1,2]), load("cr_all", [0,1,2])
rows = []
for f in FAMS:
    lo_ = load("lofo_" + f, LOFO_SEEDS)
    if lo_ is None or erm is None or cra is None: continue
    m = (fam == f); ww = w*m
    est = lambda x: (ww*x).sum()/ww.sum()
    d = lo_ - erm; vals = []
    for _ in range(1000):
        cnt = np.bincount(rng.integers(0, nk, nk), minlength=nk)[inv]; a = ww*cnt; vals.append((a*d).sum()/a.sum())
    lo, hi = np.percentile(vals, [2.5, 97.5])
    gain_seen = est(erm) - est(cra); gain_held = est(erm) - est(lo_)
    rows.append(dict(family=f, n_eval_pairs=int(m.sum()), FR_erm=round(est(erm),4), FR_cr_all_family_seen=round(est(cra),4), FR_lofo_family_held_out=round(est(lo_),4),
                     delta_lofo_vs_erm=round(est(d),4), ci_lo=round(lo,4), ci_hi=round(hi,4), sig=bool(lo>0 or hi<0),
                     share_of_gain_retained=round(gain_held/gain_seen,3) if gain_seen > 0.003 else None))
L = pd.DataFrame(rows); print(L.to_string(index=False)); L.to_csv(f"{OUT_DIR}/lofo_by_family.csv", index=False)
with zipfile.ZipFile(f"{OUT_DIR}/lofo_tables.zip", "w") as z: z.write(f"{OUT_DIR}/lofo_by_family.csv", "lofo_by_family.csv")
print("\nsend me:", f"{OUT_DIR}/lofo_tables.zip (small; results are also on Drive)")